##Подготовка поисковых запросов для медиаконтента

__Цель__: Исследовать, как разговорные формулировки и контекст диалога влияют на поиск медиаконтента, и реализовать базовый пайплайн преобразования пользовательских запросов в самостоятельные поисковые формулировки.

__Как подготавливались запросы и каталог контента__: В качестве предметной области выбран кинематограф. Сформировано 17 разнородных пользовательских сценариев, покрывающих 5 репрезентативных категорий поискового поведения:
  1. Поиск по теме, настроению или ситуации.
  2. Разговорные и неполные формулировки.
  3. Уточнения с опорой на предыдущие реплики.
  4. Ограничения и исключения.
  5. Запросы, которым переформулировка не требуется, для проверки устойчивости к избыточному вмешательству).
  
Для каждого сценария зафиксированы исходная реплика, история диалога, истинное намерение пользователя и список релевантных фильмов из каталога.
Сформирован каталог из 25 разножанровых фильмов. Для каждого объекта зафиксированы ключевые атрибуты. Текстовое представление для поиска строилось как композитное текстовое поле, объединяющее название, жанровые теги, режиссера, актерский состав, синопсис и год выпуска, что обеспечивает единое поле поиска для полнотекстовых и векторных моделей.

__Какие методы использовались и почему__:
1. LLM-based перефраз (`minimax/minimax-m3.1-flash:free` через OpenRouter API): Выбран так как задача разрешения анафор, устранения эллипсисов и восстановления пропущенных сущностей требует глубокого семантического понимания контекста диалога и общих знаний о мире.
2. Лексический поиск - BM25: Выбран так как это распространённый бейзлайн полнотекстового поиска. Чувствителен к точному вхождению именованных сущностей, вследствие чего служит индикатором того, насколько переформулировка помогает насытить запрос ключевыми униграммами.
3. Семантический поиск - Dense Retrieval: Выбран так как это компактная мультиязычная модель для кодирования предложений в плотные векторные представления. Позволяет оценить способность алгоритма сопоставлять абстрактный интент с описанием фильма.

__Какие эксперименты были проведены__:
* A/B-тестирование качества поиска: Сравнение результатов выдачи по 4 конфигурациям: BM25(Raw Query), BM25(Rewritten Query), Dense(Raw Query), Dense (Rewritten Query).
* Сравнение метрик качества ранжирования: Hit@1, MRR.
* Качественный анализ: Построчная экспертная оценка всех сценариев по сохранению смысла, корректности использования контекста и появлению галлюцинаций, неподтвержденных деталей.

__Какие выводы можно сделать для поиска медиаконтента, какие
ограничения и проблемы были обнаружены__:
1. Необходимость переформулирования запросов в диалоговых системах: Если пользователь продолжает разговор фразами вроде "а есть что-то посвежее?", обычный поиск ничего не понимает, так как в вопросе нет названий и жанров. Когда модель расшифровывает местоимения и подставляет тему из прошлых реплик, качество поиска вырастает.
2. Важно называть вещи своими именами: Если пользователь описывает сюжет своими словами, поиск часто путается. Но как только модель переводит это в точное название, нужный фильм встает на первое место.
3. Длинные описания вредят: Когда модель вместо краткой фразы выводит пересказ сюжета, поиск по ключевым словам начинает сбоить. Из-за лишних фраз система находит случайные неподходящие фильмы. Запрос должен состоять из нескольких точных слов.
4. Умный поиск не понимает отрицания: Поиск по смыслу (через числовые векторы) находит похожие вещи, но не замечает запретов.
5. Одного текста мало - нужны фильтры: Нельзя решать все задачи только изменением текста запроса. Надежная система должна работать в два шага: Текстом искать настроение, тему и суть сюжета, а годы выпуска, имена актеров и запреты отсекать отдельно.

In [5]:
import json
import requests
from typing import List, Optional
from google.colab import userdata
from openai import OpenAI
from pydantic import BaseModel, Field
import pandas as pd
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
!pip install rank-bm25 sentence-transformers scikit-learn

__Необходимые библиотеки__

In [2]:
scenarios = [
    {
        "category": "поиск по теме, настроению или ситуации",
        "dialogue_history": [],
        "raw_query": "что-нибудь фоновое, ненапряжное под готовку ужина",
        "user_intent": "Легкая комедия, ситком или кулинарное кино без сложного сюжета",
        "expected_rewrite": "легкая ненапряжная комедия повседневность для фонового просмотра"
    },
    {
        "category": "поиск по теме, настроению или ситуации",
        "dialogue_history": [],
        "raw_query": "кино чтобы прямо порыдать в конце",
        "user_intent": "Драма или мелодрама с трагическим эмоциональным финалом",
        "expected_rewrite": "эмоциональная драма с грустным финалом"
    },
    {
        "category": "поиск по теме, настроению или ситуации",
        "dialogue_history": [],
        "raw_query": "напряженный клаустрофобный триллер в одной комнате",
        "user_intent": "Триллер/детектив в формате камерного действия",
        "expected_rewrite": "камерный триллер действие в замкнутом пространстве"
    },
    {
        "category": "поиск по теме, настроению или ситуации",
        "dialogue_history": [],
        "raw_query": "что глянуть компанией под пиццу чтобы поржать",
        "user_intent": "Популярная динамичная молодежная комедия для просмотра в компании",
        "expected_rewrite": "смешная динамичная комедия для компании друзей"
    },
    {
        "category": "разговорные и неполные формулировки",
        "dialogue_history": [],
        "raw_query": "ну тот фильм где мужик во сне внутри сна ворует мысли",
        "user_intent": "Поиск конкретного фильма по размытому описанию сюжета (Начало)",
        "expected_rewrite": "фильм где проникают в чужие сны кража идей Начало Кристофер Нолан"
    },
    {
        "category": "разговорные и неполные формулировки",
        "dialogue_history": [],
        "raw_query": "криминальный боевик от гая ричи где много стреляют",
        "user_intent": "Фильм в фирменном стиле британского криминального кино Гая Ричи",
        "expected_rewrite": "криминальная комедия боевик режиссер Гай Ричи"
    },
    {
        "category": "разговорные и неполные формулировки",
        "dialogue_history": [],
        "raw_query": "сайфай с майндфаком и временными петлями",
        "user_intent": "Научно-фантастический фильм со сложной нелинейной структурой времени",
        "expected_rewrite": "научная фантастика временная петля нелинейный сюжет"
    },
    {
        "category": "разговорные и неполные формулировки",
        "dialogue_history": [],
        "raw_query": "оскароносный корейский фильм про бедняков в подвале",
        "user_intent": "Поиск фильма Паразиты Пон Чжун Хо по социальным атрибутам",
        "expected_rewrite": "фильм Паразиты Пон Чжун Хо корейская драма бедняки подвал Оскар"
    },
    {
        "category": "уточнения с опорой на предыдущие реплики",
        "dialogue_history": [
            {"role": "user", "content": "Посоветуй неонуарные детективы."},
            {"role": "assistant", "content": "Рекомендую «Бегущий по лезвию 2049», «Китайский квартал», «Семь»."}
        ],
        "raw_query": "а есть что-то посвежее из этого?",
        "user_intent": "Неонуарные детективы последних лет (современные релизы)",
        "expected_rewrite": "современные неонуарные детективные фильмы новинки"
    },
    {
        "category": "уточнения с опорой на предыдущие реплики",
        "dialogue_history": [
            {"role": "user", "content": "Какие лучшие фильмы у Кристофера Нолана?"},
            {"role": "assistant", "content": "«Темный рыцарь», «Интерстеллар», «Начало», «Престиж»."}
        ],
        "raw_query": "а с кем из них он снял фильм про фокусников?",
        "user_intent": "Фильм Нолана про фокусников (Престиж) и его актерский состав",
        "expected_rewrite": "фильм Кристофера Нолана про фокусников Престиж актеры в главных ролях"
    },
    {
        "category": "уточнения с опорой на предыдущие реплики",
        "dialogue_history": [
            {"role": "user", "content": "Ищу драмы про космос."},
            {"role": "assistant", "content": "Отличные варианты: «Интерстеллар», «Гравитация», «К звездам»."}
        ],
        "raw_query": "покажи еще похожие, но снятые в 70-80х годах",
        "user_intent": "Космическая фантастика и драмы периода 1970–1989 годов",
        "expected_rewrite": "научно-фантастические космические драмы фильмы 1970 1980 годов"
    },
    {
        "category": "уточнения с опорой на предыдущие реплики",
        "dialogue_history": [
            {"role": "user", "content": "Хочу посмотреть «Дюну» Дени Вильнёва."},
            {"role": "assistant", "content": "Прекрасный выбор, масштабный сай-фай по роману Фрэнка Герберта."}
        ],
        "raw_query": "а что он еще крутого снял в фантастике?",
        "user_intent": "Другие научно-фантастические фильмы режиссера Дени Вильнёва",
        "expected_rewrite": "фантастические фильмы режиссера Дени Вильнёва"
    },
    {
        "category": "ограничения и исключения",
        "dialogue_history": [
            {"role": "user", "content": "Посоветуй мистические фильмы на вечер."},
            {"role": "assistant", "content": "Подойдут «Заклятие», «Астрал», «Синистер»."}
        ],
        "raw_query": "только без скримеров и не сплошные ужасы, а скорее детектив",
        "user_intent": "Мистический детектив без скримеров и хоррора",
        "expected_rewrite": "мистический детектив расследование без скримеров не ужасы"
    },
    {
        "category": "ограничения и исключения",
        "dialogue_history": [],
        "raw_query": "космическая фантастика кроме интерстеллара и марсианина",
        "user_intent": "Космический сай-фай за вычетом фильмов Интерстеллар и Марсианин",
        "expected_rewrite": "научная фантастика про космос без фильмов Интерстеллар и Марсианин"
    },
    {
        "category": "ограничения и исключения",
        "dialogue_history": [
            {"role": "user", "content": "Ищу французские комедии."},
            {"role": "assistant", "content": "Классика вроде «1+1», «Амели», фильмов с Луи де Фюнесом."}
        ],
        "raw_query": "давай только современные, строго после 2015 года",
        "user_intent": "Современные французские комедии, вышедшие после 2015 года",
        "expected_rewrite": "французские комедии после 2015 года современные"
    },
    {
        "category": "переформулировка не требуется",
        "dialogue_history": [],
        "raw_query": "Интерстеллар 2014 смотреть",
        "user_intent": "Прямой поиск фильма по точному названию и году",
        "expected_rewrite": "Интерстеллар 2014 смотреть"
    },
    {
        "category": "переформулировка не требуется",
        "dialogue_history": [],
        "raw_query": "аниме Хаяо Миядзаки список",
        "user_intent": "Поиск фильмографии конкретного режиссера-аниматора",
        "expected_rewrite": "аниме Хаяо Миядзаки список"
    }
]


class FilterConstraints(BaseModel):
    exclude_titles: List[str] = Field(default_factory=list, description="Названия фильмов для исключения")
    exclude_genres: List[str] = Field(default_factory=list, description="Нежелательные жанры или тропы")
    year_min: Optional[int] = Field(None, description="Минимальный год релиза, если указан")
    year_max: Optional[int] = Field(None, description="Максимальный год релиза, если указан")


class RewriteResponse(BaseModel):
    is_rewritten: bool = Field(..., description="Флаг: была ли изменена исходная формулировка")
    standalone_query: str = Field(..., description="Автономный поисковый запрос без анафор и эллипсисов")
    reasoning: str = Field(..., description="Краткое объяснение логики трансформации")
    extracted_filters: FilterConstraints = Field(default_factory=FilterConstraints)


prompt = """Ты - интеллектуальный модуль Query Reformulation для поисковой системы фильмов и медиаконтента.
Твоя задача: превратить реплику пользователя и контекст диалога в единый САМОДОСТАТОЧНЫЙ поисковый запрос (standalone query), пригодный для текстового (BM25) и семантического (эмбеддинги) поиска.
ПРАВИЛА ТРАНСФОРМАЦИИ:
1. РАЗРЕШЕНИЕ КОНТЕКСТА: Заменяй местоимения и анафоры на конкретные сущности из истории (режиссеры, актеры, жанры, франшизы).
2. СОХРАНЕНИЕ НАМЕРЕНИЯ И ОГРАНИЧЕНИЙ: Если пользователь ищет фильм по описанию сюжета - добавь ключевые сущности сюжета и вероятное название (если оно очевидно). Все отрицания и исключения обязательно оставляй в тексте запроса и фиксируй в extracted_filters.
3. НОРМАЛИЗАЦИЯ И РАЗГОВОРНАЯ РЕЧЬ: Удаляй паразитные речевые конструкции. Преобразуй разговорные описания в устоявшиеся кинотермины.
4. КОНТРОЛЬ NO-OP: Если запрос пользователя УЖЕ является конкретным названием фильма или точной самодостаточной формулировкой без эллипсисов, НЕ меняй его смысл, не нагромождай синонимы. Выстави is_rewritten = false.
5. ФОРМАТ ВЫВОДА: Верни СТРОГО JSON-объект в соответствии со схемой:
   {
     "is_rewritten": true/false,
     "standalone_query": "...",
     "reasoning": "...",
     "extracted_filters": {
       "exclude_titles": [],
       "exclude_genres": [],
       "year_min": null,
       "year_max": null
     }
   }
"""

__Тестовые сценарии, Pydantic-схемы, промпт__
* `scenarios` - список тестовых кейсов. Включает типы запросов:·       
  * поиск по теме, настроению или ситуации
  * разговорные и неполные формулировки
  * уточнения с опорой на предыдущие реплики
  * ограничения и исключения
  * переформулировка не требуется
* `prompt` - промпт для LLM: правила трансформации запроса в standalone и требования к JSON-выводу.
* `FilterConstraints` - схема ограничений поиска: `exclude_titles`, `exclude_genres`, `year_min`, `year_max`.
* `RewriteResponse` - схема ответа модуля: `is_rewritten`, `standalone_query`, `reasoning`, `extracted_filters` (вложенный `FilterConstraints`).

In [9]:
res = requests.get("https://openrouter.ai/api/v1/models").json()
free_m = [m["id"] for m in res["data"] if ":free" in m["id"]]
for m in free_m:
    print(m)

recursive/flint-v1.0:free
minimax/minimax-m3.1-flash:free
xingchen-agi/xing4.0-29b-a4b:free
nvidia/nemotron-3.5-super-vl:free
dots-studio/dots-3-note-preview:free
liquid/lfm-2.5-2.6b:free
nvidia/nemotron-3.5-lightning:free
thinkingmachines/inkling-small:free
poolside/laguna-s-2.1:free
thinkingmachines/inkling:free
poolside/laguna-xs-2.1:free
cohere/north-mini-code:free
nvidia/nemotron-3.5-content-safety:free
nvidia/nemotron-3-ultra-550b-a55b:free
nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
google/gemma-4-26b-a4b-it:free
google/gemma-4-31b-it:free
nvidia/nemotron-3-super-120b-a12b:free


__Поиск доступных бесплатных моделей на OpenRouter__

* Запрашиваем каталог моделей OpenRouter и смотрим, что доступно без оплаты.
* Результат используется для выбора модели в следующей ячейке.

In [3]:
api_key = userdata.get('API_KEY')
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

def rewrite_query(client: OpenAI, raw_query: str, dialogue_history=None) -> RewriteResponse:
    history = dialogue_history or []
    formatted_history = "\n".join(
        f"{turn['role'].capitalize()}: {turn['content']}" for turn in history
    ) or "Контекст диалога отсутствует"

    response = client.chat.completions.create(
        model="minimax/minimax-m3.1-flash:free",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": (
                f"КОНТЕКСТ ДИАЛОГА:\n{formatted_history}\n\n"
                f"ТЕКУЩАЯ РЕПЛИКА ПОЛЬЗОВАТЕЛЯ:\n\"{raw_query}\"\n\n"
                f"Сформируй самостоятельный поисковый запрос в формате JSON."
            )},
        ],
        response_format={"type": "json_object"},
    )
    return RewriteResponse(**json.loads(response.choices[0].message.content))


__Клиент OpenRouter, функция переформулировки запроса__

* `api_key` - ключ API.
* `client` - экземпляр `OpenAI`, настроенный на `base_url="https://openrouter.ai/api/v1"`.
* `rewrite_query()` - отправляет текущую реплику и историю диалога в LLM с системным `prompt`; парсит JSON-ответ.
  * `dialogue_history=None` трактуется как пустой список; при отсутствии истории подставляется строка `"Контекст диалога отсутствует"`.
  * принудительный JSON-вывод.

In [4]:
results= []
for s in scenarios:
    res = rewrite_query(client, s["raw_query"], s["dialogue_history"])
    results.append({
        "category": s["category"],
        "raw_query": s["raw_query"],
        "rewritten_query": res.standalone_query,
        "is_rewritten": res.is_rewritten,
        "filters": res.extracted_filters.model_dump(),
        "reasoning": res.reasoning
    })

df_rewrites = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
df_rewrites

,category,raw_query,rewritten_query,is_rewritten,filters,reasoning
0,"поиск по теме, настроению или ситуации","что-нибудь фоновое, ненапряжное под готовку ужина","Фоновые фильмы и сериалы под готовку ужина: лёгкие, спокойные, не напряжённые, с уютной атмосферой и кулинарными сценами",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}",Контекст отсутствует; разговорное пожелание преобразовано в самостоятельный запрос с уточнением формата фонового просмотра и настроения: расслабляющий контент для готовки ужина.
1,"поиск по теме, настроению или ситуации",кино чтобы прямо порыдать в конце,"фильмы с душераздирающей трагичной концовкой, от которых хочется заплакать",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}","Разговорная формулировка нормализована в конкретный поисковый запрос: фильм с эмоциональной, вызывающей слёзы концовкой. Других ограничений в реплике нет."
2,"поиск по теме, настроению или ситуации",напряженный клаустрофобный триллер в одной комнате,"напряжённый клаустрофобный триллер, действие которого полностью разворачивается в одной комнате",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}",Разговорное описание нормализовано в устойчивую киноформулировку; ключевые признаки — напряжённый клаустрофобный триллер и действие в одной комнате — сохранены.
3,"поиск по теме, настроению или ситуации",что глянуть компанией под пиццу чтобы поржать,"смешные комедийные фильмы для компании на пицце-вечеринке, чтобы посмеяться",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}",Разговорный запрос преобразован в конкретный запрос на комедийные фильмы для группового просмотра; очевидное название фильма не указано.
4,разговорные и неполные формулировки,ну тот фильм где мужик во сне внутри сна ворует мысли,"Inception (Начало, 2010) — фильм Кристофера Нолана с Леонардо ДиКаприо, который проникает в чужие сны и крадёт мысли, используя многоуровневую структуру снов.",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}","Описание соответствует фильму «Inception» («Начало»): герой проникает в сны и извлекает из них идеи. Название, режиссёр и актёр добавлены для однозначного поиска."
5,разговорные и неполные формулировки,криминальный боевик от гая ричи где много стреляют,"Смерть насилием (Dead Man, 1995) — криминальный боевик Гая Ричи в жанре неонового нуара с множеством перестрелок и большим количеством стрельбы",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}","Разговорное описание преобразовано в конкретный поисковый запрос: указан вероятный фильм «Смерть насилием», режиссёр Гая Ричи, жанр и характерный признак — многочисленные перестрелки."
6,разговорные и неполные формулировки,сайфай с майндфаком и временными петлями,"научно-фантастический фильм с временными петлями, интеллектуальным сюжетом и неожиданным психологическим твистом, меняющим восприятие реальности",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': None, 'year_max': None}",Разговорные термины «сайфай» и «майндфак» нормализованы в «научно-фантастический фильм» и «интеллектуальный сюжет с неожиданным твистом»; ключевое ограничение — временные петли.
7,разговорные и неполные формулировки,оскароносный корейский фильм про бедняков в подвале,"Оскароносный корейский фильм «Паразиты» (2019) режиссёра Пон Чон Хо — драма о бедной семье, живущей в полуподвале",True,"{'exclude_titles': [], 'exclude_genres': [], 'year_min': 2019, 'year_max': 2019}","Описание однозначно указывает на «Паразитов» Пон Чон Хо; конкретное название, год, режиссёр и сюжетные детали добавлены для самостоятельного поиска."
8,уточнения с опорой на предыдущие реплики,а есть что-то посвежее из этого?,"современные неонуарные детективы, вышедшие после 2017 года",True,"{'exclude_titles': ['Бегущий по лезвию 2049', 'Китайский квартал', 'Семь'], 'exclude_genres': [], 'year_min': 2018, 'year_

__Прогон сценариев через `rewrite_query`__

* `results` - список словарей с результатами по каждому сценарию.
* `df_rewrites` - `pd.DataFrame` из `results`; отображается в конце ячейки.
* Идёт итерация по всем тестовым кейсам; для каждого вызывается `rewrite_query()`.
* Отключена обрезка длинных строк в таблице.


###Анализ результатов переформулировки запросов

Сопоставим исходные запросы и полученные формулировки. Проведём анализ по 3 вопросам:
1. Сохранение смысла и ограничений пользователя
2. Корректность использования контекста
3. Появление неподтверждённых деталей

__Тип запросов: поиск по теме, настроению или ситуации__
* Строка 0:
  * Сохранение смысла и ограничений: Основной интент (фоновое, ненапряжное) сохранён.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Появилась неподтверждённая сюжетная деталь - требование "с кулинарными сценами". Пользователь упомянул готовку как фоновую бытовую ситуацию, а не как тематику самого фильма.
  * **Итог: преобразование неудачное**

* Строка 1:
  * Сохранение смысла и ограничений: Смысл сохранён. Эмоциональный запрос нормализован в кинематографические дескрипторы ("фильмы трагичной концовкой").
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 2:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Присутствует деталь что действие должно полностью разворачиваться в одной комнате. Фильмы редко обладают таким свойством.
  * **Итог: преобразование средне-удачное**

* Строка 3:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Разговорная лексика переведена в целевой жанр ("смешные комедийные фильмы").
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Незначительное сужение формулировки до "на пицце-вечеринке". Пользователь упомянул пиццу как атрибут совместного времяпрепровождения, однако на семантику поиска комедий для компании это критически не влияет.
  * **Итог: преобразование удачное**

__Тип запросов: разговорные и неполные формулировки__
* Строка 4:
  * Сохранение смысла и ограничений: Намерение пользователя разгадано: модель идентифицировала фильм.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Формулировка перегружена избыточными подробностями (описание многоуровневой структуры снов). Запрос трансформировался в аннотацию.
  * **Итог: преобразование неудачное**

* Строка 5:
  * Сохранение смысла и ограничений: Смысл исходного запроса (поиск криминальных боевиков Гая Ричи) искажён из-за ложной привязки к конкретному фильму.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Модель приписала Гаю Ричи фильм "Dead Man" ("Мертвец" Джима Джармуша, ошибочно переведённый как "Смерть насилием"). Вместо поиска по фильмографии режиссёра запрос был ошибочно сужен до несуществующей картины.
  * **Итог: преобразование неудачное**

* Строка 6:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Сленговый термин был понят корректно.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 7:
  * Сохранение смысла и ограничений: Намерение пользователя разгадано: модель идентифицировала фильм. Модель выставила точный фильтр по годам в метаданных.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Минимальная избыточность (включение режиссёра и краткого пересказа сюжета), что оправдано для гарантированного нахождения целевого тайтла.
  * **Итог: преобразование удачное**

__Тип запросов: уточнения с опорой на предыдущие реплики__

* Строка 8:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Ограничение по новизне отражено в `year_min: 2018`.
  * Контекст: Контекст использован удачно. Модель извлекла жанр из первого шага диалога, вычислила дату самого свежего предложенного фильма  и корректно перенесла все ранее рекомендованные фильмы в фильтр исключений.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 9:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Запрос нацелен на поиск конкретного фильма и исполнителей главных ролей.
  * Контекст: Контекст использован удачно. Анафора "он" раскрыта как Кристофер Нолан, фильм про фокусников определён как "Престиж", а уточнение "с кем из них" разрешено именами ключевых актёров.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 10:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. В фильтрах зафиксированы строгие границы: `year_min: 1970, year_max: 1989`.
  * Контекст: Контекст использован удачно. унаследована тема "космические драмы и научно-фантастические фильмы", а ссылка "похожие" подкреплена упоминанием тайтлов из предыдущей реплики ассистента.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 11:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Обсуждаемый фильм добавлен в список исключений: `exclude_titles: ['Дюна']`.
  * Контекст: Контекст использован удачно. Местоимение "он" разрешено в Дени Вильнёва. Исключение "кроме Дюны" выведено из контекста беседы.
  * Неподтверждённые детали: Модель допустила частичную подмену поиска ответом, напрямую добавив в строку запроса названия конкретных картин, вместо того чтобы оставить поиск открытым для всей фильмографии режиссёра.
  * **Итог: преобразование неудачное**

__Тип запросов: ограничения и исключения__
* Строка 12:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены.
  * Контекст: Контекст использован удачно. Из предыдущей реплики унаследован контекст просмотра мистических фильмов.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 13:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

* Строка 14:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены.Временное ограничение переведено в фильтр.
  * Контекст: Контекст использован неудачно. Из диалога корректно подтянут целевой жанр и страна производства. Ранее упомянутые фильмы ("1+1", "Амели") в этот раз не были добавлены в `exclude_titles`.
  * Неподтверждённые детали: В тексте рефраза присутствует тавтология, однако на смысл и извлечённые фильтры это негативно не повлияло.
  * **Итог: преобразование неудачное**

__Тип запросов: переформулировка не требуется (No-op)__
* Строка 15:
  * Сохранение смысла и ограничений: Смысл и ключевые параметры сохранены. Запрос не подвергся искажению, флаг `is_rewritten: False` выставлен верно. В метаданных зафиксирован год.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Отсутствуют. Служебное слово "смотреть" оставлено, запрос не раздут синонимами.
  * **Итог: преобразование удачное**

* Строка 16:
  * Сохранение смысла и ограничений: Запрос оставлен в исходном виде, флаг `is_rewritten: False` выставлен корректно.
  * Контекст: Контекст отсутствовал.
  * Неподтверждённые детали: Отсутствуют.
  * **Итог: преобразование удачное**

###Типы запросов, вызывающие сложности.

1. **Бытовые ситуации и внешние условия**. Модель склонна переносить внешнее окружение пользователя в свойства контента, из-за чего происходит искусственное заужение выдачи: отсекаются подходящие фильмы.
2. **Категориальные запросы с признаками**. Склонность LLM к стремлению угадать одно конкретное название вместо составления запроса на срез каталога из-за чего может произойти ложная привязка к случайному фильму.
3. **Описания сюжета по памяти**. Сложность соблюдения формата поисковой строки. Модель пишет связный пересказ вместо набора плотных униграмм и биграмм.Из-за этого возникает шум в лексическом поиске из-за служебных слов и падения относительной частоты ключевых терминов.
4. **Комплексные отрицания**. Несоответствие между естественным языком и структурой БД: модель пишет в exclude_genres описательные фразы вроде «непрерывный хоррор», которых нет в таблице жанров.

In [9]:
movies_catalog = [
    {
        "id": 1,
        "title": "Начало",
        "year": 2010,
        "director": "Кристофер Нолан",
        "cast": ["Леонардо ДиКаприо", "Джозеф Гордон-Левитт", "Эллиот Пейдж", "Том Харди"],
        "genres": ["фантастика", "боевик", "триллер"],
        "overview": "Профессиональный вор Кобб крадет секреты из глубин подсознания во время сна. Ему предстоит выполнить обратную задачу: внедрить чужую идею в разум наследника корпорации."
    },
    {
        "id": 2,
        "title": "Престиж",
        "year": 2006,
        "director": "Кристофер Нолан",
        "cast": ["Кристиан Бэйл", "Хью Джекман", "Скарлетт Йоханссон", "Майкл Кейн"],
        "genres": ["драма", "детектив", "триллер", "фантастика"],
        "overview": "Лондон на рубеже веков. Два виртуозных иллюзиониста и фокусника ведут ожесточенную вражду, пытаясь разгадать главный секрет идеального телепортационного трюка."
    },
    {
        "id": 3,
        "title": "Остров проклятых",
        "year": 2010,
        "director": "Мартин Скорсезе",
        "cast": ["Леонардо ДиКаприо", "Марк Руффало", "Бен Кингсли"],
        "genres": ["детектив", "триллер", "психологическое"],
        "overview": "Два судебных пристава отправляются на остров в закрытую лечебницу для душевнобольных преступников, чтобы расследовать исчезновение пациентки. Но маршала преследуют галлюцинации."
    },
    {
        "id": 4,
        "title": "Паразиты",
        "year": 2019,
        "director": "Пон Чжун Хо",
        "cast": ["Сон Кан Хо", "Ли Сон Гюн", "Чхве У Шик"],
        "genres": ["драма", "триллер", "комедия"],
        "overview": "Бедная корейская семья живет в сыром полуподвале и перебивается случайными заработками. Сын устраивается репетитором в богатый дом, после чего семья обманом проникает в особняк."
    },
    {
        "id": 5,
        "title": "Кловерфилд, 10",
        "year": 2016,
        "director": "Дэн Трахтенберг",
        "cast": ["Мэри Элизабет Уинстэд", "Джон Гудман", "Джон Галлахер мл."],
        "genres": ["триллер", "фантастика", "детектив"],
        "overview": "Очнувшись после аварии, девушка понимает, что находится в запертом подземном бункере. Мужчина уверяет, что спас ее, а внешний мир разрушен химической атакой."
    },
    {
        "id": 6,
        "title": "Большой куш",
        "year": 2000,
        "director": "Гай Ричи",
        "cast": ["Джейсон Стэйтем", "Брэд Питт", "Бенисио Дель Торо", "Винни Джонс"],
        "genres": ["криминал", "комедия", "боевик"],
        "overview": "Криминальный Лондон, нелегальный бокс, похищенный гигантский бриллиант, цыгане и колоритные бандиты. Фирменный стиль Гая Ричи с динамичным монтажом и ироничными диалогами."
    },
    {
        "id": 7,
        "title": "Интерстеллар",
        "year": 2014,
        "director": "Кристофер Нолан",
        "cast": ["Мэттью Макконахи", "Энн Хэтэуэй", "Джессика Честейн"],
        "genres": ["фантастика", "драма", "приключения"],
        "overview": "Земля на грани вымирания от засухи. Группа исследователей отправляется через червоточину в дальний космос в поисках нового дома для человечества."
    },
    {
        "id": 8,
        "title": "Марсианин",
        "year": 2015,
        "director": "Ридли Скотт",
        "cast": ["Мэтт Дэймон", "Джессика Честейн", "Кристен Уиг"],
        "genres": ["фантастика", "драма", "приключения"],
        "overview": "Астронавт Марк Уотни случайно оставлен командой на Марсе во время бури. Имея ограниченные припасы, он должен выжить на непригодной планете и подать сигнал Земле."
    },
    {
        "id": 9,
        "title": "Бегущий по лезвию 2049",
        "year": 2017,
        "director": "Дени Вильнёв",
        "cast": ["Райан Гослинг", "Харрисон Форд", "Ана де Армас"],
        "genres": ["фантастика", "детектив", "неонуар"],
        "overview": "Офицер полиции Кей — репликант и охотник за головами в неоновом дождливом Лос-Анджелесе. Случайная находка ставит под угрозу хрупкий мир между людьми и машинами."
    },
    {
        "id": 10,
        "title": "Прибытие",
        "year": 2016,
        "director": "Дени Вильнёв",
        "cast": ["Эми Адамс", "Джереми Реннер", "Форест Уитакер"],
        "genres": ["фантастика", "драма", "детектив"],
        "overview": "Загадочные космические корабли пришельцев зависают над Землей. Военные привлекают лингвиста Луизу Бэнкс, чтобы найти способ общения с внеземным разумом."
    },
    {
        "id": 11,
        "title": "Семь",
        "year": 1995,
        "director": "Дэвид Финчер",
        "cast": ["Брэд Питт", "Морган Фриман", "Кевин Спейси"],
        "genres": ["триллер", "криминал", "детектив", "неонуар"],
        "overview": "В мрачном дождливом городе два детектива идут по следу изощренного серийного убийцы, который наказывает своих жертв за семь смертных грехов."
    },
    {
        "id": 12,
        "title": "12 разгневанных мужчин",
        "year": 1957,
        "director": "Сидни Люмет",
        "cast": ["Генри Фонда", "Ли Джей Кобб", "Мартин Болсам"],
        "genres": ["драма", "детектив"],
        "overview": "Двенадцать присяжных заседателей закрыты в душной комнащаве суда. Им предстоит решить судьбу юноши. Камерная психологическая драма, где действие не покидает одной комнаты."
    },
    {
        "id": 13,
        "title": "Связь",
        "year": 2013,
        "director": "Джеймс Уорд Биркит",
        "cast": ["Эмили Бальдони", "Мори Стерлинг", "Николя Браун"],
        "genres": ["фантастика", "триллер", "детектив"],
        "overview": "Компания друзей собирается на ужин в ночь пролета кометы. Внезапно гаснет свет, и реальность распадается на альтернативные параллельные ветки."
    },
    {
        "id": 14,
        "title": "Патруль времени",
        "year": 2014,
        "director": "Майкл Спириг, Питер Спириг",
        "cast": ["Итан Хоук", "Сара Снук", "Ноа Тейлор"],
        "genres": ["фантастика", "триллер", "детектив"],
        "overview": "Агент бюро времени путешествует сквозь десятилетия, чтобы предотвращать теракты. Сложная нелинейная история с парадоксальными временными петлями и шокирующим финалом."
    },
    {
        "id": 15,
        "title": "1+1",
        "year": 2011,
        "director": "Оливье Накаш, Эрик Толедано",
        "cast": ["Франсуа Клюзе", "Омар Си"],
        "genres": ["комедия", "драма", "биография"],
        "overview": "Богатый аристократ, прикованный к инвалидному креслу, нанимает сиделкой парня с криминальным прошлым. Теплая французская жизнеутверждающая комедия."
    },
    {
        "id": 16,
        "title": "Безумные преподы",
        "year": 2013,
        "director": "Пьер-Франсуа Мартен-Лаваль",
        "cast": ["Кристиан Клавье", "Изабель Нанти", "Кев Адамс"],
        "genres": ["комедия"],
        "overview": "Французская школьная комедия о худшем лицее страны, куда нанимают самых отбитых учителей, чтобы спасти выпускной экзамен."
    },
    {
        "id": 17,
        "title": "Зависнуть в Палм-Спрингс",
        "year": 2020,
        "director": "Макс Барбаков",
        "cast": ["Энди Сэмберг", "Кристин Милиоти", "Дж.К. Симмонс"],
        "genres": ["комедия", "фантастика", "мелодрама"],
        "overview": "Беззаботный парень и подружка невесты застревают во временной петле на чужой свадьбе в курортном Палм-Спрингс. Смешное и легкое кино."
    },
    {
        "id": 18,
        "title": "Осень в Нью-Йорке",
        "year": 2000,
        "director": "Джоан Чэнь",
        "cast": ["Ричард Гир", "Вайнона Райдер", "Энтони ЛаПалья"],
        "genres": ["драма", "мелодрама"],
        "overview": "Владелец ресторана влюбляется в молодую неизлечимо больную девушку. Меланхоличная романтическая драма на фоне уютных осенних пейзажей Центрального парка."
    },
    {
        "id": 19,
        "title": "Хатико: Самый верный друг",
        "year": 2009,
        "director": "Лассе Халльстрём",
        "cast": ["Ричард Гир", "Джоан Аллен"],
        "genres": ["драма", "семейный", "биография"],
        "overview": "Трогательная история верного пса, который каждый день встречал своего хозяина-профессора на вокзале, и продолжил ждать его даже после его смерти. Известная слезовыжималка."
    },
    {
        "id": 20,
        "title": "Унесенные призраками",
        "year": 2001,
        "director": "Хаяо Миядзаки",
        "cast": ["Руми Хираги", "Мию Ирино", "Мари Нацуки"],
        "genres": ["аниме", "мультфильм", "фэнтези", "приключения"],
        "overview": "Девочка Тихиро попадает в потусторонний мир духов и колдунов. Знаменитый оскароносный шедевр японской анимации режиссера Хаяо Миядзаки."
    },
    {
        "id": 21,
        "title": "Солярис",
        "year": 1972,
        "director": "Андрей Тарковский",
        "cast": ["Донатас Банионис", "Наталья Бондарчук", "Юри Ярвет"],
        "genres": ["фантастика", "драма", "детектив"],
        "overview": "Психолог прилетает на исследовательскую станцию у загадочной планеты-океана Солярис, чтобы разобраться в странных явлениях, сводящих космонавтов с ума. Классическая космическая драма 70-х."
    },
    {
        "id": 22,
        "title": "Космическая одиссея 2001 года",
        "year": 1968,
        "director": "Стэнли Кубрик",
        "cast": ["Кир Дулли", "Гэри Локвуд", "Уильям Сильвестр"],
        "genres": ["фантастика", "приключения"],
        "overview": "Экипаж космического корабля с бортовым суперкомпьютером HAL 9000 исследует происхождение загадочного черного монолита в космосе."
    },
    {
        "id": 23,
        "title": "Алиби.ком 2",
        "year": 2023,
        "director": "Филипп Лашо",
        "cast": ["Филипп Лашо", "Элоди Фонтан", "Тарек Будали"],
        "genres": ["комедия"],
        "overview": "Современная французская комедия положений. Грег закрыл свое агентство по созданию фальшивых алиби, но перед свадьбой вынужден нанять фиктивных родителей."
    },
    {
        "id": 24,
        "title": "Другие",
        "year": 2001,
        "director": "Алехандро Аменабар",
        "cast": ["Николь Кидман", "Фионнула Флэнаган"],
        "genres": ["детектив", "триллер", "мистика"],
        "overview": "Грейс живет с двумя светочувствительными детьми в уединенном особняке на острове. Атмосферный мистический детектив без скримеров и лишней жестокости с неожиданным финалом."
    },
    {
        "id": 25,
        "title": "Суперперцы",
        "year": 2007,
        "director": "Грег Моттола",
        "cast": ["Джона Хилл", "Майкл Сера", "Кристофер Минц-Плассе"],
        "genres": ["комедия"],
        "overview": "Два старшеклассника-неудачника пытаются достать выпивку для грандиозной вечеринки. Культовая безумная молодежная комедия для веселой компании."
    }
]

relevance = {0: [17, 15, 23],    # Фоновое ->                    Палм-Спрингс, 1+1, Алиби.ком 2
            1: [19, 18],        # Порыдать ->                   Хатико, Осень в Нью-Йорке
            2: [12, 5, 13],     # В 1 комнате ->                12 разгневанных мужчин, Кловерфилд 10, Связь
            3: [25, 6, 23],     # Под пиццу ->                  Суперперцы, Большой куш, Алиби.ком 2
            4: [1],             # Сон внутри сна ->             Начало
            5: [6],             # Гай Ричи перестрелки ->       Большой куш
            6: [14, 13, 17],    # Майндфак и петли ->           Патруль времени, Связь, Палм-Спрингс
            7: [4],             # Корейский бедняки ->          Паразиты
            8: [9],             # Неонуары посвежее ->          Бегущий по лезвию 2049
            9: [2],             # Нолан про фокусников ->       Престиж
            10: [21, 22],       # Космос 70-80х ->              Солярис, Космическая одиссея 2001
            11: [9, 10],        # Фантастика кроме Дюны ->      Бегущий по лезвию 2049, Прибытие
            12: [24],           # Мистика без скримеров ->      Другие
            13: [21, 22, 10],   # Космос кроме Интерстеллара -> Солярис, Одиссея, Прибытие
            14: [23],           # Комедии после 2015 ->         Алиби.ком 2
            15: [7],            # Интерстеллар 2014 ->          Интерстеллар
            16: [20]            # Миядзаки список ->            Унесенные призраками
            }

__Каталог медиаконтента с названиями и описаниями из открытых источников. Релевантность фильмов запросам__

In [10]:
def get_reciprocal_rank(ranked_ids, target_ids):
    for r, doc in enumerate(ranked_ids, 1):
        if doc in target_ids:
            return 1.0 / r
    return 0.0

def get_hit_at_k(ranked_ids, target_ids, k=1):
    return 1.0 if any(doc in target_ids for doc in ranked_ids[:k]) else 0.0

__Метрики ранжирования RR и Hit@k__

* `get_reciprocal_rank()` - reciprocal rank: 1 / позиция первого документа из `target_ids` в `ranked_ids` (позиции с 1); 0, если совпадений нет.
* `get_hit_at_k()` - Hit@k: 1, если хотя бы один `target_ids` попал в топ-k выдачи, иначе 0.
* `ranked_ids` - упорядоченный список id документов (результат поиска).
* `target_ids` - список релевантных id (эталон).
* `k` - глубина отсечения для Hit@k (по умолчанию 1).

In [11]:
corpus_texts = [f"{m['title']}. Жанры: {', '.join(m['genres'])}. Режиссер: {m['director']}. Актеры: {', '.join(m['cast'])}. Сюжет: {m['overview']} Год: {m['year']}" for m in movies_catalog]

tokenized_corpus = [doc.lower().split() for doc in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
doc_embeddings = encoder.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)

def search_bm25(query: str, top_k: int = 5):
    scores = bm25.get_scores(query.lower().split())
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{"id": movies_catalog[idx]["id"], "title": movies_catalog[idx]["title"], "year": movies_catalog[idx]["year"], "score": float(scores[idx])} for idx in top_indices]

def search_dense(query: str, top_k: int = 5):
    query_vec = encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    sims = cosine_similarity(query_vec, doc_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [{"id": movies_catalog[idx]["id"], "title": movies_catalog[idx]["title"], "year": movies_catalog[idx]["year"], "score": float(sims[idx])} for idx in top_indices]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

__Построение поисковых индексов (BM25 и dense) и функции поиска__

* `corpus_texts` - текстовые документы по каждому фильму из `movies_catalog`.
* `tokenized_corpus` - lowercase-токенизация и split по пробелам для BM25.
* `bm25` - `BM25Okapi`-индекс.
* `encoder` - `SentenceTransformer()` для получения эмбеддингов.
* `doc_embeddings` - нормализованные эмбеддинги, используются для dense-поиска.
* `search_bm25()` - BM25-поиск, сортировка по убыванию, возврат top_k.
* `search_dense()` - dense-поиск: косинусное сходство с doc_embeddings, возврат top_k.

In [12]:
experiment_rows = []
for idx, row in df_rewrites.iterrows():
    raw_q = row["raw_query"]
    rew_q = row["rewritten_query"]
    targets = relevance.get(idx, [])

    bm25_raw = search_bm25(raw_q, top_k=5)
    bm25_rew = search_bm25(rew_q, top_k=5)
    dense_raw = search_dense(raw_q, top_k=5)
    dense_rew = search_dense(rew_q, top_k=5)
    b_raw_ids = [r["id"] for r in bm25_raw]
    b_rew_ids = [r["id"] for r in bm25_rew]
    d_raw_ids = [r["id"] for r in dense_raw]
    d_rew_ids = [r["id"] for r in dense_rew]

    experiment_rows.append({
        "raw_query": raw_q,
        "bm25_raw_top1": bm25_raw[0]["title"] if bm25_raw else "-",
        "bm25_rew_top1": bm25_rew[0]["title"] if bm25_rew else "-",
        "dense_raw_top1": dense_raw[0]["title"] if dense_raw else "-",
        "dense_rew_top1": dense_rew[0]["title"] if dense_rew else "-",
        "bm25_raw_rr": get_reciprocal_rank(b_raw_ids, targets),
        "bm25_rew_rr": get_reciprocal_rank(b_rew_ids, targets),
        "dense_raw_rr": get_reciprocal_rank(d_raw_ids, targets),
        "dense_rew_rr": get_reciprocal_rank(d_rew_ids, targets),
        "bm25_raw_hit1": get_hit_at_k(b_raw_ids, targets, k=1),
        "bm25_rew_hit1": get_hit_at_k(b_rew_ids, targets, k=1),
        "dense_raw_hit1": get_hit_at_k(d_raw_ids, targets, k=1),
        "dense_rew_hit1": get_hit_at_k(d_rew_ids, targets, k=1),
    })

df_experiments = pd.DataFrame(experiment_rows)

summary = pd.DataFrame([
    {
        "Метод поиска": "BM25 (Лексический)",
        "Hit@1 (До)": f"{df_experiments['bm25_raw_hit1'].mean():.2%}",
        "Hit@1 (После)": f"{df_experiments['bm25_rew_hit1'].mean():.2%}",
        "MRR (До)": f"{df_experiments['bm25_raw_rr'].mean():.3f}",
        "MRR (После)": f"{df_experiments['bm25_rew_rr'].mean():.3f}",
        "d MRR": f"{(df_experiments['bm25_rew_rr'].mean() - df_experiments['bm25_raw_rr'].mean()):+.3f}"
    },
    {
        "Метод поиска": "Dense (Семантический)",
        "Hit@1 (До)": f"{df_experiments['dense_raw_hit1'].mean():.2%}",
        "Hit@1 (После)": f"{df_experiments['dense_rew_hit1'].mean():.2%}",
        "MRR (До)": f"{df_experiments['dense_raw_rr'].mean():.3f}",
        "MRR (После)": f"{df_experiments['dense_rew_rr'].mean():.3f}",
        "d MRR": f"{(df_experiments['dense_rew_rr'].mean() - df_experiments['dense_raw_rr'].mean()):+.3f}"
    }
])

display(summary)
display(df_experiments[["raw_query", "bm25_raw_top1", "bm25_rew_top1", "dense_raw_top1", "dense_rew_top1"]])

,Метод поиска,Hit@1 (До),Hit@1 (После),MRR (До),MRR (После),d MRR
0,BM25 (Лексический),41.18%,58.82%,0.475,0.676,+0.201
1,Dense (Семантический),29.41%,41.18%,0.407,0.590,+0.183


,raw_query,bm25_raw_top1,bm25_rew_top1,dense_raw_top1,dense_rew_top1
0,"что-нибудь фоновое, ненапряжное под готовку ужина",Бегущий по лезвию 2049,Бегущий по лезвию 2049,Паразиты,Связь
1,кино чтобы прямо порыдать в конце,Остров проклятых,Интерстеллар,Связь,Семь
2,напряженный клаустрофобный триллер в одной комнате,12 разгневанных мужчин,12 разгневанных мужчин,12 разгневанных мужчин,Космическая одиссея 2001 года
3,что глянуть компанией под пиццу чтобы поржать,"Кловерфилд, 10",Остров проклятых,Паразиты,Суперперцы
4,ну тот фильм где мужик во сне внутри сна ворует мысли,12 разгневанных мужчин,Остров проклятых,Начало,Начало
5,криминальный боевик от гая ричи где много стреляют,Большой куш,Большой куш,Семь,Семь
6,сайфай с майндфаком и временными петлями,Патруль времени,Другие,Космическая одиссея 2001 года,Начало
7,оскароносный корейский фильм про бедняков в подвале,Унесенные призраками,Паразиты,Паразиты,Паразиты
8,а есть что-то посвежее из этого?,Начало,Бегущий по лезвию 2049,Связь,Бегущий по лезвию 2049
9,а с кем из них он снял фильм про фокусников?,Марсианин,Престиж,Престиж,Солярис


### Сравнение результатов для исходных и преобразованных запросов.

В рамках эксперимента сопоставлена работа двух подходов к информационному поиску - лексического и семантического.

__Количественные результаты эксперимента__: Переформулировка запросов обеспечила прирост качества по обеим парадигмам поиска:
* BM25 показал прирост +17.64% по метрике Hit@1 и +0.201 по MRR за счет насыщения запроса точными именованными сущностями (актеры, режиссеры, названия франшиз).
* Dense Retrieval показал прирост +11.77% по метрике Hit@1 и +0.183 по MRR благодаря раскрытию контекстных ссылок.

__Cоответствие найденного контента намерению пользователя__:
1. Уточнения с опорой на предыдущие реплики: Запросы вида "а есть что-то посвежее из этого?" или "а с кем из них он снял фильм про фокусников?" возвращали случайный шум. После рефраза выдача стала строго соответствовать намерению: BM25 вывел на первое место "Бегущий по лезвию 2049" и "Престиж".
2. Описания сюжета по памяти: В строке 7 сырой BM25 выдал "Унесенные призраками", тогда как после добавления названия "Паразиты" оба метода вывели точно.
3. Разговорные и неполные формулировки, поиск по теме, настроению или ситуации: В строке 3 сырой поиск выдавал триллеры. Преобразование запроса в термины молодежной комедии позволило Dense-поиску поднять на 1-е место целевой фильм.

__Случаи, когда преобразование радикально помогло__:
* Устранение эллипсисов и анафор (Строки 8, 9, 10):
  * Строка 8: BM25 и Dense поднялся с мусорного результата к релевантному.
  * Строка 9: BM25 сменил нерелевантный "Марсианин" на целевой "Престиж".
  * Строка 10: Dense-поиск по рефразу смог найти классику научной фантастики.
* Идентификация сущностей по описаниям (Строка 7):
  * Включение названия "Паразиты" исправило ошибку BM25.
* Фильтрация по эпохе и новизне (Строка 14):
  * Запрос в сыром виде выдавал "Марсианина" и "Палм-Спрингс". Семантический поиск по рефразу вывел на первое место современную французскую комедию.

__Случаи, когда преобразование ухудшило результат__:
* Избыточная многословность (Строка 4):
  * BM25 запутался в распределении весов TF-IDF и вывел на первое место "Остров проклятых" из-за частого упоминания связки ДиКаприо/триллер.
* Галлюцинации (Строка 6):
  * Сырой BM25 безошибочно нашел фильм про петли. После переформулировки с абстрактными понятиями "интеллектуальный сюжет, меняющий восприятие реальности" BM25 сместился на мистический детектив.
* Неспособность эмбеддингов обрабатывать отрицания (Строка 13):
  * И до, и после преобразования Dense-поиск вывел на 1-е место "Марсианин". Векторные модели ориентируются на семантическую близость понятий: токены "Марсианин" и "Интерстеллар" обладают наибольшим весом, а частица "кроме" игнорируется.

__Ограничения проведённой проверки__:

1. Каталог содержит небольшое число документов, поэтому вероятность случайных ложноположительных совпадений по общеупотребительным словам высокая.
2. Модуль переформулировки извлекал структурированные фильтры, однако в эксперименте поиск выполнялся исключительно по сгенерированной текстовой строке. Из-за этого потенциал извлеченных метаданных не был задействован.
3. Использовалась универсальная мультиязычная модель. Модель плохо сопоставляет бытовые ситуации ("под готовку ужина", "под пиццу") с кинематографическими тегами.
4. Использовалось базовое разбиение по пробелам без лемматизации и стемминга слов русского языка. Это снизило потенциал BM25 на сырых запросах.